In [6]:
# usearch aynı klasörde, teyit etmek için çağırıyoruz. ! Shell komutu olarak çalıştırmak içindir.
!usearch

usearch v11.0.667_win32, 2.0Gb RAM (16.6Gb total), 12 cores
(C) Copyright 2013-18 Robert C. Edgar, all rights reserved.
https://drive5.com/usearch

License: personal use only



In [13]:
import pandas as pd
import os

# csv dosyası yüklenir.
df = pd.read_csv("file_name.csv")

# Dizileri yazmak için yeni bir FASTA dosyası açılır.
with open("file_name.fasta", "w") as fasta_file:
    for index, row in df.iterrows():
        # Antikanser özelliğe ait değeri saklayacak şekilde başlık düzenlenir.
        fasta_file.write(f'>seq{index}_anticancer_{row["anticancer"]}\n')
        # Sekans yazılır.
        fasta_file.write(f'{row["sequence"]}\n')

In [14]:
!usearch -cluster_fast file_name.fasta -id 0.5 -centroids file_name-uc.fasta

00:00 4.8Mb     0.1% Reading merged-main.fasta
00:00 4.9Mb   100.0% Reading merged-main.fasta

00:00 4.9Mb  CPU has 12 cores, defaulting to 10 threads
00:00 5.3Mb     2.3% DF
00:00 5.3Mb   100.0% DF

00:00 5.4Mb     0.3% DB
00:00 5.4Mb   100.0% DB

00:00 44Mb      0.3% Clustering
00:00 44Mb      0.3% 0 clusters, max size 0, avg 0.0
00:00 53Mb    100.0% 314 clusters, max size 10, avg 1.3

00:00 53Mb    0.3% Writing centroids to merged-main-uc.fasta
00:00 53Mb    100.0% Writing centroids to merged-main-uc.fasta

                                                              
      Seqs  396
  Clusters  314
  Max size  10
  Avg size  1.3
  Min size  1
Singletons  288, 72.7% of seqs, 91.7% of clusters
   Max mem  53Mb
      Time  1.00s
Throughput  396.0 seqs/sec.



usearch v11.0.667_win32, 2.0Gb RAM (16.6Gb total), 12 cores
(C) Copyright 2013-18 Robert C. Edgar, all rights reserved.
https://drive5.com/usearch

License: personal use only



In [15]:
from Bio import SeqIO
import csv

# FASTA dosyası yüklenir.
fasta_file = "file_name-uc.fasta"

# Çıktı dosyasının adı ve/veya dizini
csv_file = "file_name-uc.csv"

# FASTA dosyasını açılır ve bir CSV dosyasına yazdırılır.
with open(fasta_file, mode="r") as fasta, open(csv_file, mode="w", newline="") as csv_out:
    csv_writer = csv.writer(csv_out)
    csv_writer.writerow(["ID", "anticancer", "sequence"])

    # FASTA dosyasındaki her kayıt ayrıştırılır ve yazılır.
    for record in SeqIO.parse(fasta, "fasta"):
        # Antikanser özelliğini ayrıca almak için
        parts = record.id.split("_")
        sequence_id = parts[0]
        anticancer_property = parts[-1]

        csv_writer.writerow([sequence_id, anticancer_property, str(record.seq)])

In [ ]:
# Veri setindeki negatif ve pozitif kayıtların tekrardan dengelenmesi gerekmektedir.

df = pd.read_csv("file_name-uc.csv")

# Antikanser değerine göre ayrılır.
df_class_0 = df[df["anticancer"] == 0]
df_class_1 = df[df["anticancer"] == 1]

# Her kümenin sayısı alınır.
count_class_0, count_class_1 = df["anticancer"].value_counts()

# En az olan küme bulunur.
minority_count = min(count_class_0, count_class_1)

# Karıştırma işlemi (bias'ı azaltmak için)
df_class_0 = df_class_0.sample(frac=1, random_state=42)
df_class_1 = df_class_1.sample(frac=1, random_state=42)

# Her iki küme de en az sayıya sahip sınıfın sayısına eşitlenir
df_class_0_under = df_class_0.sample(minority_count, random_state=42)
df_class_1_under = df_class_1.sample(minority_count, random_state=42)

# Birleştirme işlemi
df_balanced = pd.concat([df_class_0_under, df_class_1_under], axis=0)

# Yeni csv dosyasına yazılır.
df_balanced.to_csv("final_file_name.csv", index=False)